# Caga librerías, datos, crea df útiles

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import itertools
from itertools import chain, combinations


In [2]:
data= pd.read_csv('X_train.csv')
pesos= pd.read_csv('y_train.csv')

In [3]:
numerados= pd.DataFrame()

numerados['edad_padre']= data['fage']
numerados['edad_madre']= data['mage']
numerados['madurez']= data['mature'].map({'younger mom':0, 'mature mom':1})
numerados['visitas_medico']= data['visits']
numerados['aumento_peso_madre']=data['gained']
numerados['sexo']= data['sex'].map({'male':0, 'female':1})
numerados['madre_fumadora']= data['habit'].map({'nonsmoker':0, 'smoker':1})
numerados['estado_civil']= data['marital'].map({'not married':0, 'married':1})
numerados['raza_madre']= data['whitemom'].map({'white':0, 'not white':1})


# Iteración entre todas las combinaciones posibles
Priorizo por el que de menor MAE

## Modo cavernícola

In [4]:
#Modelo basado en el aumento de peso de la madre

x= np.array(data['gained']).reshape(-1,1)
y=np.array(pesos['weight'])

reg= LinearRegression()
reg.fit(x,y)
y_pred= reg.predict(x)
MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

print(f'Modelo basado en gained {MAE}')

Modelo basado en gained 0.9051583564168633


In [5]:
def ajuste_univariado(parametro):
    x= np.array(numerados[parametro]).reshape(-1,1)
    y=np.array(pesos['weight'])

    reg= LinearRegression()
    reg.fit(x,y)
    y_pred= reg.predict(x)
    MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

    print(f'ajustando con {parametro} el MAE resulta: {MAE}')


In [6]:
for parametro in numerados.columns:
    ajuste_univariado(parametro)


ajustando con edad_padre el MAE resulta: 0.9086638558663798
ajustando con edad_madre el MAE resulta: 0.9087635776416253
ajustando con madurez el MAE resulta: 0.9092777059246197
ajustando con visitas_medico el MAE resulta: 0.9129169435397145
ajustando con aumento_peso_madre el MAE resulta: 0.9051583564168633
ajustando con sexo el MAE resulta: 0.9022654896077402
ajustando con madre_fumadora el MAE resulta: 0.9104535303488892
ajustando con estado_civil el MAE resulta: 0.9104185679012344
ajustando con raza_madre el MAE resulta: 0.9094842446719082


In [7]:
def ajuste_parvariado(parametro1, parametro2):
        x= np.array(numerados[[parametro1, parametro2]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        print(f'ajustando con {parametro1} y {parametro2} el MAE resulta: {MAE}')

In [8]:
for parametro1 in numerados.columns:
    for parametro2 in numerados.columns:
       ajuste_parvariado(parametro1, parametro2)

ajustando con edad_padre y edad_padre el MAE resulta: 0.9086638558663799
ajustando con edad_padre y edad_madre el MAE resulta: 0.9097659420680652
ajustando con edad_padre y madurez el MAE resulta: 0.9084154970755614
ajustando con edad_padre y visitas_medico el MAE resulta: 0.9091639209628031
ajustando con edad_padre y aumento_peso_madre el MAE resulta: 0.9022127909877542
ajustando con edad_padre y sexo el MAE resulta: 0.8991199026841905
ajustando con edad_padre y madre_fumadora el MAE resulta: 0.9077285126550116
ajustando con edad_padre y estado_civil el MAE resulta: 0.9082071775750081
ajustando con edad_padre y raza_madre el MAE resulta: 0.9038992020135771
ajustando con edad_madre y edad_padre el MAE resulta: 0.9097659420680652
ajustando con edad_madre y edad_madre el MAE resulta: 0.9087635776416253
ajustando con edad_madre y madurez el MAE resulta: 0.9084399950016309
ajustando con edad_madre y visitas_medico el MAE resulta: 0.9089694571347778
ajustando con edad_madre y aumento_peso_m

In [9]:
def ajuste_trivariado(parametro1, parametro2, parametro3):
        x= np.array(numerados[[parametro1, parametro2, parametro3]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {parametro1} y {parametro2} el MAE resulta: {MAE}')
        return MAE

In [10]:
maes=[]

for parametro1 in numerados.columns:
    for parametro2 in numerados.columns:
       for parametro3 in numerados.columns:
             MAE= ajuste_trivariado(parametro1, parametro2, parametro3)

             tupla=(parametro1,parametro2,parametro3,MAE)
             maes.append(tupla)

maes.sort(key=lambda x: x[3])
print(maes)


[('aumento_peso_madre', 'sexo', 'raza_madre', 0.8894190818007952), ('aumento_peso_madre', 'raza_madre', 'sexo', 0.8894190818007952), ('sexo', 'aumento_peso_madre', 'raza_madre', 0.8894190818007952), ('raza_madre', 'aumento_peso_madre', 'sexo', 0.8894190818007952), ('raza_madre', 'sexo', 'aumento_peso_madre', 0.8894190818007952), ('sexo', 'raza_madre', 'aumento_peso_madre', 0.8894190818007953), ('madurez', 'aumento_peso_madre', 'sexo', 0.8924514363519359), ('madurez', 'sexo', 'aumento_peso_madre', 0.8924514363519359), ('aumento_peso_madre', 'madurez', 'sexo', 0.8924514363519359), ('aumento_peso_madre', 'sexo', 'madurez', 0.8924514363519359), ('sexo', 'madurez', 'aumento_peso_madre', 0.8924514363519359), ('sexo', 'aumento_peso_madre', 'madurez', 0.8924514363519359), ('edad_madre', 'aumento_peso_madre', 'sexo', 0.8925868767744009), ('edad_madre', 'sexo', 'aumento_peso_madre', 0.8925868767744009), ('aumento_peso_madre', 'edad_madre', 'sexo', 0.8925868767744009), ('aumento_peso_madre', 'sex

In [11]:
def ajuste_cuatrivariado(parametro1, parametro2, parametro3, parametro4):
        x= np.array(numerados[[parametro1, parametro2, parametro3, parametro4]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {parametro1} y {parametro2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
maes=[]

for parametro1 in numerados.columns:
    for parametro2 in numerados.columns:
       for parametro3 in numerados.columns:
             for parametro4 in numerados.columns:
                MAE= ajuste_cuatrivariado(parametro1, parametro2, parametro3, parametro4)

                tupla=(parametro1,parametro2,parametro3,parametro4,MAE)
                maes.append(tupla)

maes.sort(key=lambda x: x[4])
print(maes)

In [ ]:
def ajuste_pentavariado(parametro1, parametro2, parametro3, parametro4, parametro5):
        x= np.array(numerados[[parametro1, parametro2, parametro3, parametro4, parametro5]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {parametro1} y {parametro2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
maes=[]

for parametro1 in numerados.columns:
    for parametro2 in numerados.columns:
       for parametro3 in numerados.columns:
             for parametro4 in numerados.columns:
                    for parametro5 in numerados.columns:
                        MAE= ajuste_pentavariado(parametro1, parametro2, parametro3, parametro4, parametro5)

                        tupla=(parametro1,parametro2,parametro3,parametro4,parametro5, MAE)
                        maes.append(tupla)

maes.sort(key=lambda x: x[5])
print(maes)

[('edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre', 0.8821243182546038), ('edad_padre', 'sexo', 'aumento_peso_madre', 'madre_fumadora', 'raza_madre', 0.8821243182546038), ('edad_padre', 'madre_fumadora', 'sexo', 'aumento_peso_madre', 'raza_madre', 0.8821243182546038), ('edad_padre', 'madre_fumadora', 'sexo', 'raza_madre', 'aumento_peso_madre', 0.8821243182546038), ('edad_padre', 'madre_fumadora', 'raza_madre', 'sexo', 'aumento_peso_madre', 0.8821243182546038), ('edad_padre', 'raza_madre', 'sexo', 'madre_fumadora', 'aumento_peso_madre', 0.8821243182546038), ('aumento_peso_madre', 'edad_padre', 'madre_fumadora', 'raza_madre', 'sexo', 0.8821243182546038), ('aumento_peso_madre', 'edad_padre', 'raza_madre', 'madre_fumadora', 'sexo', 0.8821243182546038), ('aumento_peso_madre', 'sexo', 'edad_padre', 'raza_madre', 'madre_fumadora', 0.8821243182546038), ('aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre', 'edad_padre', 0.8821243182546038), ('aumento_peso_madr

In [ ]:
prueba= list(itertools.combinations(numerados.columns,5))

print(prueba)

[('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre'), ('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'sexo'), ('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'madre_fumadora'), ('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'estado_civil'), ('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'raza_madre'), ('edad_padre', 'edad_madre', 'madurez', 'aumento_peso_madre', 'sexo'), ('edad_padre', 'edad_madre', 'madurez', 'aumento_peso_madre', 'madre_fumadora'), ('edad_padre', 'edad_madre', 'madurez', 'aumento_peso_madre', 'estado_civil'), ('edad_padre', 'edad_madre', 'madurez', 'aumento_peso_madre', 'raza_madre'), ('edad_padre', 'edad_madre', 'madurez', 'sexo', 'madre_fumadora'), ('edad_padre', 'edad_madre', 'madurez', 'sexo', 'estado_civil'), ('edad_padre', 'edad_madre', 'madurez', 'sexo', 'raza_madre'), ('edad_padre', 'edad_madre', 'madurez', 'madre_fumadora', 'estado_civil'), ('edad_padre', 'edad_madre', 'madurez', 'madre_f

In [ ]:
combinaciones= list(itertools.combinations(numerados.columns,5))

maes=[]

for i in combinaciones:
    MAE= ajuste_pentavariado(i[0], i[1], i[2], i[3], i[4])
    tupla=(i, MAE)
    maes.append(tupla)

maes.sort(key=lambda x: x[1])
print(maes)

[(('edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8821243182546038), (('madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8841928699987843), (('edad_madre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8843594245453287), (('edad_padre', 'madurez', 'aumento_peso_madre', 'sexo', 'raza_madre'), 0.8855131411742655), (('edad_padre', 'aumento_peso_madre', 'sexo', 'estado_civil', 'raza_madre'), 0.8859321326028169), (('madurez', 'aumento_peso_madre', 'sexo', 'estado_civil', 'raza_madre'), 0.8861605493939957), (('aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8864785907871643), (('madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'raza_madre'), 0.8865733053265541), (('visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8867394124282314), (('edad_padre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'raza_madre'), 0.8869212776398723), (('edad_madre

In [ ]:
def ajuste_sextuvariado(p1, p2, p3, p4, p5, p6):
        x= np.array(numerados[[p1, p2, p3, p4, p5, p6]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
combinaciones= list(itertools.combinations(numerados.columns,6))

maes=[]

for i in combinaciones:
    MAE= ajuste_sextuvariado(i[0], i[1], i[2], i[3], i[4], i[5])
    tupla=(i, MAE)
    maes.append(tupla)

maes.sort(key=lambda x: x[1])
print(maes)

[(('edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8823343575765121), (('edad_padre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8825925354808881), (('edad_padre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8832618154195838), (('edad_madre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8836410149081493), (('madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8838917612231368), (('madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8845172816433353), (('edad_madre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8852795926430277), (('edad_madre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8852967587195552), (('edad_padre', 'edad_madre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'ra

In [ ]:
def ajuste_heptavariado(p1, p2, p3, p4, p5, p6, p7):
        x= np.array(numerados[[p1, p2, p3, p4, p5, p6, p7]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
combinaciones= list(itertools.combinations(numerados.columns,7))

maes=[]

for i in combinaciones:
    MAE= ajuste_heptavariado(i[0], i[1], i[2], i[3], i[4], i[5], i[6])
    tupla=(i, MAE)
    maes.append(tupla)

maes.sort(key=lambda x: x[1])
print(maes)

[(('edad_padre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8826040361702735), (('edad_padre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8827238146346136), (('edad_padre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8834378297300898), (('edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8839674950558957), (('madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8841097595755492), (('edad_madre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8845178665397428), (('edad_padre', 'edad_madre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8846279056403684), (('edad_padre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'estado_civil', 'raza_

In [ ]:
def ajuste_octavariado(p1, p2, p3, p4, p5, p6, p7, p8):
        x= np.array(numerados[[p1, p2, p3, p4, p5, p6, p7, p8]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
combinaciones= list(itertools.combinations(numerados.columns,8))

maes=[]

for i in combinaciones:
    MAE= ajuste_octavariado(i[0], i[1], i[2], i[3], i[4], i[5], i[6], i[7])
    tupla=(i, MAE)
    maes.append(tupla)

maes.sort(key=lambda x: x[1])
print(maes)

[(('edad_padre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8826031488494583), (('edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8845397841067407), (('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8846112339718342), (('edad_padre', 'edad_madre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8852701141606455), (('edad_padre', 'edad_madre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8860384458191479), (('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'estado_civil', 'raza_madre'), 0.887832219535971), (('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil'), 0.89100636983

In [ ]:
def ajuste_novenovariado(p1, p2, p3, p4, p5, p6, p7, p8, p9):
        x= np.array(numerados[[p1, p2, p3, p4, p5, p6, p7, p8, p9]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
combinaciones= list(itertools.combinations(numerados.columns,9))

maes=[]

for i in combinaciones:
    MAE= ajuste_novenovariado(i[0], i[1], i[2], i[3], i[4], i[5], i[6], i[7], i[8])
    tupla=(i, MAE)
    maes.append(tupla)

maes.sort(key=lambda x: x[1])
print(maes)

[(('edad_padre', 'edad_madre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8850488132131297)]


In [ ]:
def ajuste_decavariado(p1, p2, p3, p4, p5, p6, p7, p8, p9, p10):
        x= np.array(numerados[[p1, p2, p3, p4, p5, p6, p7, p8, p9, p10]])
        y=np.array(pesos['weight'])

        reg= LinearRegression()
        reg.fit(x,y)
        y_pred= reg.predict(x)
        MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción

        #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
        return MAE

In [ ]:
combinaciones= list(itertools.combinations(numerados.columns,10))

maes=[]

for i in combinaciones:
    MAE= ajuste_decavariado(i[0], i[1], i[2], i[3], i[4], i[5], i[6], i[7], i[8], i[9])
    tupla=(i, MAE)
    maes.append(tupla)

maes.sort(key=lambda x: x[1])
print(maes)

[]


## Modo civilizado

In [4]:
def ajuste(combo):
    x= np.array(numerados[list(combo)])
    y=np.array(pesos['weight'])

    reg= LinearRegression()
    reg.fit(x,y)
    y_pred= reg.predict(x)
    MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción
    R2= reg.score(x,y)

    #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
    return MAE, R2

In [7]:
def combinaciones(lista):
    s=list(lista)
    return chain.from_iterable(combinations(s,r) for r in range(1, len(s) + 1))

In [11]:
maes=[]
erres=[]

for combo in combinaciones(numerados.columns):
    MAE, R2= ajuste(combo)
    tupla=(combo, MAE)
    tupla_r2=(combo, R2)
    maes.append(tupla)
    erres.append(tupla_r2)

maes.sort( key=lambda x: x[1])
print(maes)
erres.sort(reverse=True, key=lambda x: x[1])
print(erres)

[(('edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.882124318254604), (('edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8823343575765121), (('edad_padre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8825925354808881), (('edad_padre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8826031488494583), (('edad_padre', 'madurez', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.8826040361702737), (('edad_padre', 'madurez', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8827238146346136), (('edad_padre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.883261815419584), (('edad_padre', 'visitas_medico', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'estado_civil', 'raza_madre'), 0.8834378297300898), (('edad_madre', 'madu

Mejor combinación: ('edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'), 0.882124318254604)

Por ahora el R2 da tan mal que mejor no darle bola.
Lo mismo con sacar el 10% para testear, esto sirve para detectar overfitting pero el modelo es tan malo que ni vale la pena, sacar 10% cuando el MAE de más bajo.

# Polinomios

In [10]:
from sklearn.preprocessing import PolynomialFeatures
polyfeats= PolynomialFeatures(degree=2, include_bias=True)
X=np.array(numerados[['edad_padre']])
X_poly= polyfeats.fit_transform(X)
print(X_poly)


[[1.000e+00 3.400e+01 1.156e+03]
 [1.000e+00 3.600e+01 1.296e+03]
 [1.000e+00 3.700e+01 1.369e+03]
 ...
 [1.000e+00 3.700e+01 1.369e+03]
 [1.000e+00 2.700e+01 7.290e+02]
 [1.000e+00 2.100e+01 4.410e+02]]


In [12]:
from sklearn.pipeline import make_pipeline

poly_reg= make_pipeline(
    PolynomialFeatures(degree=2, include_bias=True),
    LinearRegression(fit_intercept=False)
)

poly_reg.fit(np.array(numerados[['edad_padre']]), pesos['weight'])

reg= poly_reg.steps[1][1]
print('Coeficientes = {}'.format(reg.coef_))

Coeficientes = [6.95291113e+00 9.46598267e-03 1.86456006e-05]


In [13]:
from sklearn.compose import ColumnTransformer

grado_independientes= ColumnTransformer([
    ('Cuadráticas', PolynomialFeatures(degree=2, include_bias=False), ['edad_padre']),
    ('Lineales', 'passthrough', ['aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'])
])

modelo= make_pipeline(grado_independientes, LinearRegression())

x=numerados[['edad_padre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre']]
y= pesos['weight']

modelo.fit(x,y)
y_pred= modelo.predict(x)
MAE= mean_absolute_error(y,y_pred)

print(MAE)

0.8821479701820893


## Interacciones

Se puede hacer con Pandas creando una nueva columna

In [27]:
numerados['I_pedad_medad']= numerados['edad_padre']*numerados['edad_madre']
combo=('I_pedad_medad', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre')

ajuste(combo)


(0.8824102048500229, 0.07532010942857248)

se puede hacer son sklearn  
pero ajusta con los términos individuales y además con el de interacción

In [30]:
grados= ColumnTransformer([
    ('Interacciones', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False), ['edad_padre', 'edad_madre']),
    ('Lineales', 'passthrough', ['aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre'])
])

modelillo= make_pipeline(grados, LinearRegression())

x=numerados[['edad_padre', 'edad_madre', 'aumento_peso_madre', 'sexo', 'madre_fumadora', 'raza_madre']]
y= pesos['weight']

modelillo.fit(x,y)
y_pred= modelo.predict(x)
MAE= mean_absolute_error(y,y_pred)

print(MAE)

0.8821479701820893


In [4]:
def ajuste_con_interacciones(combo_lineal, combo_interaccion):
    grados= ColumnTransformer([
    ('Interacciones', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False), combo_interaccion),
    ('Lineales', 'passthrough', combo_lineal)
    ])

    modelillo= make_pipeline(grados, LinearRegression())

    x= np.array(numerados[list(combo_interaccion + combo_lineal)])
    y=np.array(pesos['weight'])

    reg= LinearRegression()
    reg.fit(x,y)
    y_pred= reg.predict(x)
    MAE= mean_absolute_error(y,y_pred) #Cuanto menor de MAE mejor la predicción
    R2= reg.score(x,y)

    #print(f'ajustando con {p1} y {p2} el MAE resulta: {MAE}')
    return MAE, R2

In [5]:
def interacciones(lista):
    s=list(lista)
    return chain.from_iterable(combinations(s,2))

In [14]:
maes=[]
erres=[]

for combo_lineal in combinaciones(numerados.columns):
    for combo_interaccion in interacciones(numerados.columns):
        MAE, R2= ajuste_con_interacciones(combo_lineal, combo_interaccion)
        tupla=(combo_lineal,combo_interaccion, MAE)
        tupla_r2=(combo_lineal,combo_interaccion, R2)
        maes.append(tupla)
        erres.append(tupla_r2)

maes.sort( key=lambda x: x[2])
print(maes)
erres.sort(reverse=True, key=lambda x: x[2])
print(erres)

TypeError: can only concatenate str (not "tuple") to str

que dificil es codear y que facil pedirle a chat cgt

In [19]:
from itertools import combinations, chain
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

def combinaciones(lista):
    s = list(lista)
    return chain.from_iterable(combinations(s, r) for r in range(1, len(s) + 1))

def interacciones(lista):
    return list(combinations(lista, 2))

maes = []
erres = []

columnas_disponibles = list(numerados.columns)

for combo_lineal in combinaciones(columnas_disponibles):
    for combo_interaccion in interacciones(columnas_disponibles):
        
        # Aseguramos que siempre sean listas, sin importar si vienen como tupla o string
        l_lineal = [combo_lineal] if isinstance(combo_lineal, str) else list(combo_lineal)
        l_inter = [combo_interaccion] if isinstance(combo_interaccion, str) else list(combo_interaccion)
        
        cols_necesarias = list(set(l_lineal) | set(l_inter))
        
        grados = ColumnTransformer([
            ('Interacciones', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False), l_inter),
            ('Lineales', 'passthrough', l_lineal)
        ])

        modelillo = make_pipeline(grados, LinearRegression())

        x = numerados[cols_necesarias]
        y = pesos['weight']

        modelillo.fit(x, y)
        y_pred = modelillo.predict(x)
        
        MAE = mean_absolute_error(y, y_pred)
        R2 = r2_score(y, y_pred)

        maes.append((l_lineal, l_inter, MAE))
        erres.append((l_lineal, l_inter, R2))

# Se corrige el índice a 2 porque el valor métrico está al final de la tupla
maes.sort(key=lambda x: x[2])
erres.sort(reverse=True, key=lambda x: x[2])
df_maes = pd.DataFrame(maes, columns=['Lineales', 'Interacciones', 'MAE'])

In [ ]:
df_erres = pd.DataFrame(erres, columns=['Lineales', 'Interacciones', 'erres'])

In [18]:
df_maes = pd.DataFrame(maes, columns=['Lineales', 'Interacciones', 'MAE'])
df_maes.head(10)

,Lineales,Interacciones,MAE
0,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
1,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
2,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
3,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
4,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
5,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
6,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
7,"[edad_padre, aumento_peso_madre, sexo, raza_ma...",[madre_fumadora],0.882124
8,"[edad_padre, aumento_peso_madre, sexo, madre_f...",[aumento_peso_madre],0.882124
9,"[edad_padre, aumento_peso_madre, sexo, madre_f...",[sexo],0.882124
